# 0D Coupled Cardiovascular Simulator — Human Engineering

Python translation of the browser-runnable model in `docs/sim/heart-0d.js`. Same physics, same parameters. Use this for parameter sweeps, batch scenarios, and research.

**Atlas entries:**
- [Heart](../../atlases/01-human/06-organ/heart/README.md)
- [Cardiovascular System](../../atlases/01-human/07-system/cardiovascular-system/README.md)
- [Calcium](../../atlases/01-human/02-atomic/calcium/README.md) — the ion whose transient is abstracted as Emax/Emin elastance

**Reference:** Smith BW, Chase JG, Nokes RI, Shaw GM, Wake G. Minimal haemodynamic system model including ventricular interaction and valve dynamics. *Med Eng Phys.* 2004;26(2):131–139. [doi:10.1016/j.medengphy.2003.10.001](https://doi.org/10.1016/j.medengphy.2003.10.001)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.integrate import solve_ivp

%matplotlib inline
plt.rcParams.update({
    'figure.facecolor': '#07120f',
    'axes.facecolor': '#0a0d14',
    'axes.edgecolor': '#1c2230',
    'axes.labelcolor': '#8ea69b',
    'xtick.color': '#8ea69b',
    'ytick.color': '#8ea69b',
    'text.color': '#ecfff7',
    'grid.color': '#1c2230',
    'grid.linewidth': 0.8,
    'lines.linewidth': 1.8,
})

## Parameters

Match exactly those in `docs/sim/heart-0d.js`. Units: mmHg, mL, seconds, mL/s.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# HEALTHY adult parameter set
# Matches HEALTHY constant in heart-0d.js
# At steady state: MAP ~93 mmHg, EF ~56%, SV ~70 mL, CO ~5.2 L/min
# ──────────────────────────────────────────────────────────────────────────────

HEALTHY = {
    'HR': 75,          # heart rate, bpm
    'Tsystole': 0.30,  # systolic duration, s

    # Ventricular chambers: Emax (mmHg/mL), Emin (mmHg/mL), V0 (mL, unstressed vol)
    'LV': {'Emax': 2.5,  'Emin': 0.06, 'V0': 15},
    'RV': {'Emax': 0.55, 'Emin': 0.04, 'V0': 10},

    # Vascular compartments: C (mL/mmHg), V0 (mL, unstressed volume)
    'Ao':  {'C': 1.0,  'V0': 30},   # systemic arteries (aorta + large arteries)
    'Pa':  {'C': 4.0,  'V0': 20},   # pulmonary arteries
    'Pv':  {'C': 25.0, 'V0': 50},   # pulmonary veins
    'Vc':  {'C': 50.0, 'V0': 100},  # systemic veins (vena cava + venules)

    # Valve resistances (mmHg·s/mL), diode model: Q = max(0, dP/R)
    'MV':  {'R': 0.006},  # mitral (LA → LV)
    'AV':  {'R': 0.005},  # aortic (LV → Ao)
    'TV':  {'R': 0.006},  # tricuspid (RA → RV) — represented via Vc→RV
    'PuV': {'R': 0.004},  # pulmonary (RV → Pa)

    'R_sys':  1.00,  # systemic vascular resistance, mmHg·s/mL
    'R_pulm': 0.10,  # pulmonary vascular resistance, mmHg·s/mL
}

# Six scenarios (patches on top of HEALTHY)
SCENARIOS = {
    'healthy': {
        'label': 'Healthy adult',
        'description': 'Resting hemodynamics, HR 75, MAP ~93 mmHg.',
        'patch': {},
    },
    'heartFailure': {
        'label': 'Heart failure (HFrEF)',
        'description': 'Reduced LV contractility; chamber dilates; EF falls.',
        'patch': {
            'LV': {'Emax': 1.0, 'Emin': 0.10, 'V0': 25},
            'R_sys': 1.40,
        },
    },
    'aorticStenosis': {
        'label': 'Aortic stenosis',
        'description': 'Stenotic AV; LV pressure rises sharply during systole.',
        'patch': {'AV': {'R': 0.025}},
    },
    'hypertension': {
        'label': 'Hypertension',
        'description': 'Elevated systemic vascular resistance.',
        'patch': {'R_sys': 1.80},
    },
    'tachycardia': {
        'label': 'Sinus tachycardia',
        'description': 'HR 130, shortened diastole, reduced filling time.',
        'patch': {'HR': 130, 'Tsystole': 0.22},
    },
    'bradycardia': {
        'label': 'Sinus bradycardia',
        'description': 'HR 45, prolonged diastole, larger SV from increased filling.',
        'patch': {'HR': 45, 'Tsystole': 0.34},
    },
}

# Initial state vector [V_lv, V_rv, V_ao, V_pa, V_pv, V_vc] in mL
INITIAL_STATE = np.array([125.0, 100.0, 80.0, 35.0, 175.0, 205.0])

print('Parameters loaded.')
print(f"Total stressed volume: {INITIAL_STATE.sum():.0f} mL")

## Model: ODE Function

State vector: `y = [V_lv, V_rv, V_ao, V_pa, V_pv, V_vc]`

Pressures are computed from volumes via time-varying elastance (ventricles) and compliance (vessels).
Valve flows are diode-like: `Q = max(0, ΔP/R)`.

Reference: Smith et al. (2004), equations 1–8.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Helper functions
# ──────────────────────────────────────────────────────────────────────────────

def apply_patch(base, patch):
    """Deep-merge patch on top of base parameter dict (one level deep for sub-dicts)."""
    import copy
    result = copy.deepcopy(base)
    for k, v in patch.items():
        if isinstance(v, dict) and k in result and isinstance(result[k], dict):
            result[k] = {**result[k], **v}
        else:
            result[k] = v
    return result


def activation(cycle_phase, systole_frac):
    """Normalized cardiac activation function, 0..1 over the cycle.
    Squared-sine pulse during systole; zero during diastole.
    Matches heart-0d.js `activation()`.
    """
    if cycle_phase >= systole_frac:
        return 0.0
    x = np.pi * cycle_phase / systole_frac
    return np.sin(x) ** 2


def valve_flow(p_up, p_dn, R):
    """Diode-like valve: forward flow only. Q = max(0, (pUp - pDn) / R)."""
    dp = p_up - p_dn
    return dp / R if dp > 0 else 0.0


def heart_odes(t, y, params):
    """ODE RHS for the 0D coupled cardiovascular model.

    Args:
        t: current time (s)
        y: state vector [V_lv, V_rv, V_ao, V_pa, V_pv, V_vc] (mL)
        params: HeartParams dict

    Returns:
        dydt: state derivative (mL/s)
    """
    p = params
    V_lv, V_rv, V_ao, V_pa, V_pv, V_vc = y

    # Cardiac cycle timing
    T = 60.0 / p['HR']                        # cycle period, s
    phase = (t % T) / T                        # fraction of cycle, 0..1
    e = activation(phase, p['Tsystole'] / T)   # activation, 0..1

    # Time-varying elastance for ventricles
    E_lv = p['LV']['Emin'] + (p['LV']['Emax'] - p['LV']['Emin']) * e
    E_rv = p['RV']['Emin'] + (p['RV']['Emax'] - p['RV']['Emin']) * e

    # Pressures (mmHg)
    P_lv = E_lv * max(V_lv - p['LV']['V0'], 0.0)
    P_rv = E_rv * max(V_rv - p['RV']['V0'], 0.0)
    P_ao = (V_ao - p['Ao']['V0']) / p['Ao']['C']
    P_pa = (V_pa - p['Pa']['V0']) / p['Pa']['C']
    P_pv = (V_pv - p['Pv']['V0']) / p['Pv']['C']
    P_vc = (V_vc - p['Vc']['V0']) / p['Vc']['C']

    # Valve flows (mL/s), diode model
    Q_mv  = valve_flow(P_pv, P_lv, p['MV']['R'])    # mitral:    pulm vein → LV
    Q_av  = valve_flow(P_lv, P_ao, p['AV']['R'])    # aortic:   LV → aorta
    Q_tv  = valve_flow(P_vc, P_rv, p['TV']['R'])    # tricuspid: syst vein → RV
    Q_pv  = valve_flow(P_rv, P_pa, p['PuV']['R'])   # pulm:     RV → pulm artery

    # Microcirculation flows (Windkessel)
    Q_sys  = (P_ao - P_vc) / p['R_sys']    # systemic bed
    Q_pulm = (P_pa - P_pv) / p['R_pulm']   # pulmonary bed

    # Volume derivatives (conservation: inflow - outflow)
    dV_lv = Q_mv  - Q_av
    dV_rv = Q_tv  - Q_pv
    dV_ao = Q_av  - Q_sys
    dV_pa = Q_pv  - Q_pulm
    dV_pv = Q_pulm - Q_mv
    dV_vc = Q_sys  - Q_tv

    return [dV_lv, dV_rv, dV_ao, dV_pa, dV_pv, dV_vc]


def compute_pressures(t, y, params):
    """Recompute all pressures and flows at a given (t, y) for post-processing."""
    p = params
    V_lv, V_rv, V_ao, V_pa, V_pv, V_vc = y
    T = 60.0 / p['HR']
    phase = (t % T) / T
    e = activation(phase, p['Tsystole'] / T)
    E_lv = p['LV']['Emin'] + (p['LV']['Emax'] - p['LV']['Emin']) * e
    E_rv = p['RV']['Emin'] + (p['RV']['Emax'] - p['RV']['Emin']) * e
    P_lv = E_lv * max(V_lv - p['LV']['V0'], 0.0)
    P_rv = E_rv * max(V_rv - p['RV']['V0'], 0.0)
    P_ao = (V_ao - p['Ao']['V0']) / p['Ao']['C']
    P_pa = (V_pa - p['Pa']['V0']) / p['Pa']['C']
    return {'P_lv': P_lv, 'P_rv': P_rv, 'P_ao': P_ao, 'P_pa': P_pa, 'e': e}


print('ODE function defined.')

## Simulation — Healthy Adult

Integrate 10 seconds (first 5 s burn-in for transient to decay, last 5 s for analysis).

In [ ]:
def run_simulation(params, t_span=(0, 10), t_eval_dt=0.002, y0=None):
    """Run the 0D heart simulation and return dense output.

    Args:
        params: HeartParams dict
        t_span: (t_start, t_end) in seconds
        t_eval_dt: output time step (s); default 2 ms
        y0: initial state (default: INITIAL_STATE)

    Returns:
        sol: scipy OdeResult with .t (times) and .y (6 × N states)
    """
    if y0 is None:
        y0 = INITIAL_STATE.copy()
    t_eval = np.arange(t_span[0], t_span[1], t_eval_dt)
    sol = solve_ivp(
        heart_odes,
        t_span,
        y0,
        args=(params,),
        method='RK45',
        t_eval=t_eval,
        rtol=1e-6,
        atol=1e-8,
        dense_output=True,
    )
    return sol


def extract_metrics(sol, params, burn_in=5.0):
    """Extract steady-state hemodynamic metrics from solution.

    Considers only time > burn_in (default 5 s) to allow transients to decay.

    Returns:
        dict with SV, EF, HR, CO, MAP keys.
    """
    mask = sol.t >= burn_in
    t_ss = sol.t[mask]
    y_ss = sol.y[:, mask]  # 6 × N

    V_lv = y_ss[0, :]
    edv = V_lv.max()
    esv = V_lv.min()
    sv = edv - esv
    ef = sv / edv if edv > 0 else 0
    hr = params['HR']
    co = sv * hr / 1000  # L/min

    # MAP: mean aortic pressure over steady-state window
    P_ao_arr = np.array([
        (y_ss[2, i] - params['Ao']['V0']) / params['Ao']['C']
        for i in range(y_ss.shape[1])
    ])
    map_val = P_ao_arr.mean()

    return {'SV_mL': sv, 'EDV_mL': edv, 'ESV_mL': esv, 'EF_pct': ef * 100,
            'HR_bpm': hr, 'CO_Lmin': co, 'MAP_mmHg': map_val}


# Run healthy simulation
print("Running healthy simulation (10 s)...")
sol_healthy = run_simulation(HEALTHY, t_span=(0, 10))
metrics_healthy = extract_metrics(sol_healthy, HEALTHY)

print("\nHealthy adult — steady-state metrics:")
for k, v in metrics_healthy.items():
    print(f"  {k:15s}: {v:.2f}")

## Plotting — Healthy Adult Hemodynamics

In [ ]:
def synthetic_ecg(cycle_phase):
    """Phenomenological ECG signal from cycle phase. Matches syntheticEcg() in heart-0d.js."""
    waves = [
        {'center': 0.05, 'amp':  0.15, 'sigma': 0.022},  # P
        {'center': 0.16, 'amp': -0.10, 'sigma': 0.005},  # Q
        {'center': 0.18, 'amp':  1.00, 'sigma': 0.006},  # R
        {'center': 0.20, 'amp': -0.20, 'sigma': 0.008},  # S
        {'center': 0.40, 'amp':  0.30, 'sigma': 0.040},  # T
    ]
    v = 0.0
    for w in waves:
        z = (cycle_phase - w['center']) / w['sigma']
        v += w['amp'] * np.exp(-0.5 * z * z)
    return v


def plot_hemodynamics(sol, params, title='Healthy adult', burn_in=5.0):
    mask = sol.t >= burn_in
    t = sol.t[mask]
    y = sol.y[:, mask]
    V_lv, V_rv, V_ao, V_pa, V_pv, V_vc = y

    T = 60.0 / params['HR']
    phases = (t % T) / T

    P_ao = (V_ao - params['Ao']['V0']) / params['Ao']['C']
    P_lv = np.array([
        (params['LV']['Emin'] + (params['LV']['Emax'] - params['LV']['Emin']) *
         activation(phases[i], params['Tsystole'] / T)) * max(V_lv[i] - params['LV']['V0'], 0)
        for i in range(len(t))
    ])
    P_pv = (V_pv - params['Pv']['V0']) / params['Pv']['C']
    ecg = np.array([synthetic_ecg(ph) for ph in phases])

    fig = plt.figure(figsize=(14, 9), facecolor='#07120f')
    gs = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.35)
    fig.suptitle(title, color='#ecfff7', fontsize=14, fontweight='bold', y=0.98)

    # Display at most 4 cycles for clarity
    t_window = min(4 * T, t[-1] - t[0])
    t0 = t[-1] - t_window
    mask2 = t >= t0
    t2 = t[mask2] - t0

    # ECG
    ax0 = fig.add_subplot(gs[0, 0])
    ax0.plot(t2, ecg[mask2], color='#34d399', linewidth=1.5)
    ax0.set_title('ECG (synthetic)', color='#8ea69b', fontsize=10)
    ax0.set_ylabel('mV (norm)', color='#8ea69b')
    ax0.set_xlabel('Time (s)', color='#8ea69b')
    ax0.grid(True, alpha=0.3)
    ax0.axhline(0, color='#253830', linewidth=0.8)

    # Pressures
    ax1 = fig.add_subplot(gs[0, 1])
    ax1.plot(t2, P_ao[mask2], color='#f87171', label='Aortic', linewidth=1.5)
    ax1.plot(t2, P_lv[mask2], color='#60a5fa', label='LV', linewidth=1.5)
    ax1.plot(t2, P_pv[mask2], color='#a78bfa', label='Pulm. vein (≈LA)', linewidth=1.0)
    ax1.set_title('Pressures (mmHg)', color='#8ea69b', fontsize=10)
    ax1.set_ylabel('Pressure (mmHg)', color='#8ea69b')
    ax1.set_xlabel('Time (s)', color='#8ea69b')
    ax1.legend(facecolor='#0a0d14', edgecolor='#1c2230', labelcolor='#ecfff7', fontsize=8)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(-5, 165)

    # Volumes
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.plot(t2, V_lv[mask2], color='#fb923c', label='LV', linewidth=1.5)
    ax2.plot(t2, V_rv[mask2], color='#22d3ee', label='RV', linewidth=1.5)
    ax2.set_title('Chamber Volumes (mL)', color='#8ea69b', fontsize=10)
    ax2.set_ylabel('Volume (mL)', color='#8ea69b')
    ax2.set_xlabel('Time (s)', color='#8ea69b')
    ax2.legend(facecolor='#0a0d14', edgecolor='#1c2230', labelcolor='#ecfff7', fontsize=8)
    ax2.grid(True, alpha=0.3)

    # PV loop
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.plot(V_lv[mask2], P_lv[mask2], color='#fb923c', linewidth=2.0)
    ax3.set_title('LV Pressure–Volume Loop', color='#8ea69b', fontsize=10)
    ax3.set_xlabel('LV Volume (mL)', color='#8ea69b')
    ax3.set_ylabel('LV Pressure (mmHg)', color='#8ea69b')
    ax3.grid(True, alpha=0.3)
    ax3.set_xlim(0, 200)
    ax3.set_ylim(0, 160)

    for ax in [ax0, ax1, ax2, ax3]:
        ax.set_facecolor('#0a0d14')
        ax.spines['bottom'].set_color('#1c2230')
        ax.spines['left'].set_color('#1c2230')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.show()


plot_hemodynamics(sol_healthy, HEALTHY, title='Healthy Adult — 0D Cardiovascular Model')

## Scenario Sweep

Compare all 6 clinical scenarios. Shows overlaid pressure and PV loop traces.

In [ ]:
SCENARIO_COLORS = {
    'healthy':       '#00d9a3',
    'heartFailure':  '#f87171',
    'aorticStenosis':'#c084fc',
    'hypertension':  '#ffb86b',
    'tachycardia':   '#3aa9ff',
    'bradycardia':   '#34d399',
}

# Run all scenarios
results = {}
for key, scenario in SCENARIOS.items():
    params_s = apply_patch(HEALTHY, scenario['patch'])
    print(f"  Running {scenario['label']}...")
    sol_s = run_simulation(params_s, t_span=(0, 10))
    metrics_s = extract_metrics(sol_s, params_s)
    results[key] = {'params': params_s, 'sol': sol_s, 'metrics': metrics_s}

print("\nAll scenarios complete.")
print(f"\n{'Scenario':<22} {'HR':>5} {'SV':>6} {'EF':>5} {'CO':>6} {'MAP':>7}")
print("-" * 55)
for key, r in results.items():
    m = r['metrics']
    print(f"{SCENARIOS[key]['label']:<22} {m['HR_bpm']:>5.0f} {m['SV_mL']:>6.1f} "
          f"{m['EF_pct']:>5.1f} {m['CO_Lmin']:>6.2f} {m['MAP_mmHg']:>7.1f}")

In [ ]:
def plot_scenario_comparison(results, burn_in=5.0):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#07120f')
    fig.suptitle('All 6 Scenarios — Pressure & PV Loop Comparison',
                 color='#ecfff7', fontsize=12, fontweight='bold')

    ax_p, ax_pv = axes

    for key, r in results.items():
        sol = r['sol']
        params = r['params']
        color = SCENARIO_COLORS[key]
        label = SCENARIOS[key]['label']

        mask = sol.t >= burn_in
        t = sol.t[mask]
        y = sol.y[:, mask]
        V_lv, V_rv, V_ao = y[0], y[1], y[2]

        T = 60.0 / params['HR']
        phases = (t % T) / T
        P_ao = (V_ao - params['Ao']['V0']) / params['Ao']['C']
        P_lv = np.array([
            (params['LV']['Emin'] + (params['LV']['Emax'] - params['LV']['Emin']) *
             activation(phases[i], params['Tsystole'] / T)) * max(V_lv[i] - params['LV']['V0'], 0)
            for i in range(len(t))
        ])

        # Show 2 cycles
        t_win = min(2 * T, t[-1] - t[0])
        t0 = t[-1] - t_win
        m2 = t >= t0
        t2 = t[m2] - t0

        ax_p.plot(t2, P_ao[m2], color=color, linewidth=1.5, label=label)
        ax_pv.plot(V_lv[m2], P_lv[m2], color=color, linewidth=1.8, label=label)

    for ax, title, xlabel, ylabel in [
        (ax_p, 'Aortic Pressure (2 cycles)', 'Time (s)', 'Pressure (mmHg)'),
        (ax_pv, 'LV Pressure–Volume Loop', 'LV Volume (mL)', 'LV Pressure (mmHg)'),
    ]:
        ax.set_facecolor('#0a0d14')
        ax.set_title(title, color='#8ea69b', fontsize=10)
        ax.set_xlabel(xlabel, color='#8ea69b')
        ax.set_ylabel(ylabel, color='#8ea69b')
        ax.grid(True, alpha=0.3, color='#1c2230')
        ax.spines['bottom'].set_color('#1c2230')
        ax.spines['left'].set_color('#1c2230')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.legend(facecolor='#0a0d14', edgecolor='#1c2230',
                  labelcolor='#ecfff7', fontsize=7, loc='best')

    ax_pv.set_xlim(0, 220)
    ax_pv.set_ylim(0, 200)

    plt.tight_layout()
    plt.show()


plot_scenario_comparison(results)

## References

1. **Smith BW, Chase JG, Nokes RI, Shaw GM, Wake G.** Minimal haemodynamic system model including ventricular interaction and valve dynamics. *Med Eng Phys.* 2004;26(2):131–139. [doi:10.1016/j.medengphy.2003.10.001](https://doi.org/10.1016/j.medengphy.2003.10.001)

2. **Bers DM.** Cardiac excitation-contraction coupling. *Nature.* 2002;415(6868):198–205. [doi:10.1038/415198a](https://doi.org/10.1038/415198a) · [PubMed 11805843](https://pubmed.ncbi.nlm.nih.gov/11805843/)

3. **OpenStax.** *Anatomy & Physiology 2e*, Ch. 19: The Cardiovascular System — The Heart. [openstax.org](https://openstax.org/books/anatomy-and-physiology-2e/pages/19-1-heart-anatomy)

4. **Heidenreich PA, Bozkurt B, Aguilar D, et al.** 2022 AHA/ACC/HFSA Guideline for the Management of Heart Failure. *Circulation.* 2022;145(18):e895–e1032. [doi:10.1161/CIR.0000000000001063](https://doi.org/10.1161/CIR.0000000000001063)